# Synthetic demo data -> Fabric lakehouse

Generic, parameterised notebook to generate a **credible** synthetic dataset and land it as
Delta tables in a Fabric lakehouse, ready for a **Fabric data agent**.

Replace the domain section with your own. Everything else (seasonality, storyline, validation,
write, verify) is reusable as-is.

**How to use it**

1. Attach this notebook to a lakehouse in your Fabric workspace.
2. Edit the **Parameters** cell below.
3. Run all. It takes about a minute for a few hundred thousand rows.
4. Read the *Verify* section out loud - those numbers are the ones your agent will be asked about.

> Runs in Fabric (Spark) and on a laptop (pandas + CSV fallback), so you can test before you demo.

## Parameters

In [ ]:
# --- Domain -----------------------------------------------------------------
DOMAIN          = "retail"          # free text, used in table comments only
TABLE_PREFIX    = ""                # e.g. "demo_" to avoid collisions

# --- Volume -----------------------------------------------------------------
N_STORES        = 12
N_PRODUCTS      = 150
N_CUSTOMERS     = 5_000
N_TRANSACTIONS  = 200_000

# --- Time window ------------------------------------------------------------
START_DATE      = "2024-01-01"
END_DATE        = "2025-12-31"

# --- Reproducibility --------------------------------------------------------
SEED            = 42

# --- Write ------------------------------------------------------------------
WRITE_MODE      = "overwrite"       # overwrite | append
DRY_RUN         = False             # True = generate and validate, write nothing

## 1. Setup

`USE_SPARK` is detected, not configured: the same notebook runs in Fabric and on a laptop.

In [ ]:
import os
import random
from datetime import date, timedelta

import numpy as np
import pandas as pd

random.seed(SEED)
np.random.seed(SEED)

try:
    spark  # noqa: F821
    USE_SPARK = True
except NameError:
    USE_SPARK = False

print(f"Spark available: {USE_SPARK}")

start = pd.Timestamp(START_DATE)
end = pd.Timestamp(END_DATE)
all_days = pd.date_range(start, end, freq="D")
print(f"{len(all_days)} days, {start.date()} -> {end.date()}")

## 2. Reference data

Dimensions first. Keep them **small and readable** - an analyst (and an agent) must be able to
list every store or category in one glance. Names must be pronounceable: you will read them out
loud during the demo.

In [ ]:
REGIONS = ["North", "South", "East", "West"]
CITIES = {
    "North": ["Lille", "Amiens", "Rouen"],
    "South": ["Marseille", "Toulouse", "Nice"],
    "East":  ["Strasbourg", "Lyon", "Dijon"],
    "West":  ["Nantes", "Rennes", "Bordeaux"],
}
CATEGORIES = {
    "Paint":      (12.0, 60.0),
    "Tools":      (8.0, 250.0),
    "Garden":     (5.0, 180.0),
    "Lighting":   (9.0, 120.0),
    "Hardware":   (1.5, 40.0),
}

# --- stores -----------------------------------------------------------------
rows = []
for i in range(N_STORES):
    region = REGIONS[i % len(REGIONS)]
    city = CITIES[region][i // len(REGIONS) % len(CITIES[region])]
    rows.append({
        "store_id": f"ST{i+1:03d}",
        "store_name": f"{city} {'Center' if i % 2 == 0 else 'Retail Park'}",
        "region": region,
        "city": city,
        "opened_on": (start - timedelta(days=int(np.random.randint(400, 3000)))).date(),
        "surface_sqm": int(np.random.choice([800, 1200, 1800, 2500])),
    })
stores = pd.DataFrame(rows)

# --- products ---------------------------------------------------------------
rows = []
for i in range(N_PRODUCTS):
    cat = list(CATEGORIES)[i % len(CATEGORIES)]
    lo, hi = CATEGORIES[cat]
    price = round(float(np.random.uniform(lo, hi)), 2)
    rows.append({
        "product_id": f"P{i+1:05d}",
        "product_name": f"{cat} item {i+1:03d}",
        "category": cat,
        "unit_price": price,
        "unit_cost": round(price * float(np.random.uniform(0.45, 0.75)), 2),
    })
products = pd.DataFrame(rows)

# --- customers --------------------------------------------------------------
SEGMENTS = ["Consumer", "Pro", "Contractor"]
customers = pd.DataFrame({
    "customer_id": [f"C{i+1:06d}" for i in range(N_CUSTOMERS)],
    "segment": np.random.choice(SEGMENTS, N_CUSTOMERS, p=[0.7, 0.2, 0.1]),
    "loyalty_tier": np.random.choice(["None", "Silver", "Gold"], N_CUSTOMERS, p=[0.6, 0.3, 0.1]),
    "signed_up_on": [
        (start - timedelta(days=int(d))).date()
        for d in np.random.randint(0, 1800, N_CUSTOMERS)
    ],
})

display(stores.head()) if USE_SPARK else print(stores.head())
print(f"{len(stores)} stores, {len(products)} products, {len(customers)} customers")

## 3. Facts, with structure

Uniform random data is what makes a demo fall flat: every question returns a flat line and the
agent looks useless. Build the signal in explicitly.

| Effect | Why |
|---|---|
| Yearly seasonality | Gives "compare Q4 to Q3" a real answer |
| Weekday profile | Makes "which day sells most" meaningful |
| Growth trend | Makes year-over-year comparisons interesting |
| Per-store multiplier | Creates a top performer and a laggard |
| Noise | Keeps the chart from looking generated |

In [ ]:
day_index = np.arange(len(all_days))

# Yearly seasonality: peak around late spring and December.
seasonality = (
    1.0
    + 0.25 * np.sin(2 * np.pi * (day_index / 365.25) - 0.6)
    + 0.15 * np.exp(-((all_days.dayofyear.values - 350) ** 2) / (2 * 12 ** 2))
)

# Weekday profile: Mon=0 ... Sun=6
WEEKDAY_FACTOR = np.array([0.85, 0.85, 0.90, 1.00, 1.25, 1.45, 0.55])
weekday = WEEKDAY_FACTOR[all_days.dayofweek.values]

# Mild growth, ~12% a year.
trend = 1.0 + 0.12 * (day_index / 365.25)

daily_weight = seasonality * weekday * trend
daily_weight = np.clip(daily_weight, 0.05, None)

# Each store has its own scale: one clear leader, one clear laggard.
store_factor = np.random.lognormal(0, 0.28, N_STORES)
store_factor[0] *= 1.6    # flagship
store_factor[-1] *= 0.55  # struggling
store_factor /= store_factor.sum()

# Pareto-ish product popularity.
product_weight = np.random.pareto(1.3, N_PRODUCTS) + 0.2
product_weight /= product_weight.sum()

print("daily weight min/max:", round(daily_weight.min(), 2), round(daily_weight.max(), 2))

In [ ]:
# Allocate transactions across days, then across stores/products/customers.
p_day = daily_weight / daily_weight.sum()
tx_day_idx = np.random.choice(len(all_days), N_TRANSACTIONS, p=p_day)
tx_store_idx = np.random.choice(N_STORES, N_TRANSACTIONS, p=store_factor)
tx_product_idx = np.random.choice(N_PRODUCTS, N_TRANSACTIONS, p=product_weight)
tx_customer_idx = np.random.randint(0, N_CUSTOMERS, N_TRANSACTIONS)

quantity = np.random.choice([1, 1, 1, 2, 2, 3, 4, 6], N_TRANSACTIONS)
unit_price = products["unit_price"].values[tx_product_idx]
unit_cost = products["unit_cost"].values[tx_product_idx]

# Discounts concentrated on Pro / Contractor segments.
segment = customers["segment"].values[tx_customer_idx]
base_discount = np.where(segment == "Contractor", 0.12, np.where(segment == "Pro", 0.07, 0.0))
discount = np.round(base_discount + np.random.choice([0, 0, 0, 0.05, 0.10], N_TRANSACTIONS), 2)

net_price = np.round(unit_price * (1 - discount), 2)
transactions = pd.DataFrame({
    "transaction_id": [f"T{i+1:08d}" for i in range(N_TRANSACTIONS)],
    "transaction_date": all_days[tx_day_idx].date,
    "store_id": stores["store_id"].values[tx_store_idx],
    "product_id": products["product_id"].values[tx_product_idx],
    "customer_id": customers["customer_id"].values[tx_customer_idx],
    "quantity": quantity,
    "unit_price": unit_price,
    "discount_pct": discount,
    "net_amount": np.round(net_price * quantity, 2),
    "margin_amount": np.round((net_price - unit_cost) * quantity, 2),
    "channel": np.random.choice(["Store", "Online", "Click&Collect"], N_TRANSACTIONS, p=[0.65, 0.25, 0.10]),
}).sort_values("transaction_date").reset_index(drop=True)

print(transactions.shape)
transactions.head()

## 4. Plant the storyline

**This is the cell that makes or breaks the demo.** A data agent is only impressive if there is
something to find. Decide the two or three questions you will ask on stage, then write the answer
into the data here.

Below: a store that collapses after a date, and a category that takes off. Both are visible in a
`GROUP BY`, which is exactly what the agent will generate.

In [ ]:
STORYLINE = []

# (a) One store drops ~45% after a given date -> "which store is underperforming, and since when?"
INCIDENT_STORE = stores["store_id"].iloc[3]
INCIDENT_FROM = pd.Timestamp("2025-06-01").date()
mask = (transactions["store_id"] == INCIDENT_STORE) & (transactions["transaction_date"] >= INCIDENT_FROM)
drop = mask & (np.random.rand(len(transactions)) < 0.45)
transactions = transactions.loc[~drop].reset_index(drop=True)
STORYLINE.append(f"{INCIDENT_STORE} loses ~45% of its volume from {INCIDENT_FROM}")

# (b) One category grows strongly in the last 6 months -> "what is driving growth?"
BOOM_CATEGORY = "Garden"
boom_products = set(products.loc[products["category"] == BOOM_CATEGORY, "product_id"])
BOOM_FROM = pd.Timestamp("2025-07-01").date()
boom_mask = transactions["product_id"].isin(boom_products) & (transactions["transaction_date"] >= BOOM_FROM)
transactions.loc[boom_mask, "quantity"] = (transactions.loc[boom_mask, "quantity"] * 2).astype(int)
transactions.loc[boom_mask, "net_amount"] = (transactions.loc[boom_mask, "net_amount"] * 2).round(2)
transactions.loc[boom_mask, "margin_amount"] = (transactions.loc[boom_mask, "margin_amount"] * 2).round(2)
STORYLINE.append(f"{BOOM_CATEGORY} doubles in volume from {BOOM_FROM}")

# (c) A handful of deep-discount transactions -> "where are we losing margin?"
loss = transactions.sample(frac=0.004, random_state=SEED).index
transactions.loc[loss, "discount_pct"] = 0.55
transactions.loc[loss, "net_amount"] = (transactions.loc[loss, "net_amount"] * 0.45).round(2)
transactions.loc[loss, "margin_amount"] = (transactions.loc[loss, "margin_amount"] * -0.3).round(2)
STORYLINE.append(f"{len(loss)} transactions sold at 55% discount, negative margin")

print("Questions your agent can now answer:")
for s in STORYLINE:
    print(" -", s)

## 5. Validate before writing

Never write data you have not looked at. Broken referential integrity or a negative total is the
fastest way to lose the room.

In [ ]:
def check(label, condition, detail=""):
    status = "OK  " if condition else "FAIL"
    print(f"[{status}] {label} {detail}")
    return condition

ok = True
ok &= check("no null in transactions", not transactions.isnull().any().any())
ok &= check("store FK integrity", transactions["store_id"].isin(stores["store_id"]).all())
ok &= check("product FK integrity", transactions["product_id"].isin(products["product_id"]).all())
ok &= check("customer FK integrity", transactions["customer_id"].isin(customers["customer_id"]).all())
ok &= check("unique transaction ids", transactions["transaction_id"].is_unique)
ok &= check("dates in window",
            (transactions["transaction_date"] >= start.date()).all()
            and (transactions["transaction_date"] <= end.date()).all())
ok &= check("revenue is positive", transactions["net_amount"].sum() > 0)
ok &= check("all stores present", transactions["store_id"].nunique() == N_STORES,
            f"({transactions['store_id'].nunique()}/{N_STORES})")

assert ok, "Fix the failures above before writing."
print("\nAll checks passed.")

## 6. Write to the lakehouse

Delta tables in Fabric, CSV locally. Same call either way.

In [ ]:
TABLES = {
    "stores": stores,
    "products": products,
    "customers": customers,
    "transactions": transactions,
}

def write_table(name: str, pdf: pd.DataFrame, mode: str = WRITE_MODE):
    table_name = f"{TABLE_PREFIX}{name}"
    if not USE_SPARK:
        os.makedirs("./out", exist_ok=True)
        path = f"./out/{table_name}.csv"
        pdf.to_csv(path, index=False)
        return f"csv -> {path}"

    sdf = spark.createDataFrame(pdf)  # noqa: F821
    try:
        sdf.write.mode(mode).format("delta").saveAsTable(table_name)
        return f"delta table -> {table_name}"
    except Exception as exc:  # managed-table name taken, or no default lakehouse
        print(f"  saveAsTable failed ({type(exc).__name__}), falling back to path write")
        sdf.write.mode(mode).format("delta").save(f"Tables/{table_name}")
        return f"delta path -> Tables/{table_name}"

if DRY_RUN:
    print("DRY_RUN = True, nothing written.")
else:
    for name, pdf in TABLES.items():
        print(f"{name:<14} {len(pdf):>8,} rows  ", write_table(name, pdf))

## 7. Verify what landed

Run the aggregates you are going to ask the agent for. If a number here surprises you, it will
surprise you on stage too.

In [ ]:
tx = transactions.copy()
tx["month"] = pd.to_datetime(tx["transaction_date"]).dt.to_period("M").astype(str)

print("== Revenue by year ==")
tx["year"] = pd.to_datetime(tx["transaction_date"]).dt.year
print(tx.groupby("year")["net_amount"].sum().round(0).to_string())

print("\n== Top 5 stores ==")
top = tx.merge(stores, on="store_id").groupby("store_name")["net_amount"].sum().nlargest(5)
print(top.round(0).to_string())

print("\n== Monthly revenue of the incident store ==")
inc = tx[tx["store_id"] == INCIDENT_STORE].groupby("month")["net_amount"].sum()
print(inc.tail(12).round(0).to_string())

print("\n== Revenue by category, last 6 months ==")
recent = tx[pd.to_datetime(tx["transaction_date"]) >= pd.Timestamp(BOOM_FROM)]
print(recent.merge(products, on="product_id").groupby("category")["net_amount"]
      .sum().sort_values(ascending=False).round(0).to_string())

print("\n== Negative-margin transactions ==")
neg = tx[tx["margin_amount"] < 0]
print(f"{len(neg):,} rows, {neg['margin_amount'].sum():,.0f} of margin lost")

## Next step: the data agent

1. In the Fabric workspace, **Create -> Data agent**.
2. Add this lakehouse as a data source and **select only the four tables above**. Fewer tables,
   better answers.
3. Give every table and every non-obvious column a **description**. The agent reads them; this is
   the single highest-leverage thing you can do.
4. Add **agent instructions**: the business domain, the fiscal calendar, what "revenue" means
   (`net_amount`, not `unit_price * quantity`), and which table to prefer for which question.
5. Add **example queries** for the three storyline questions. Examples beat instructions.
6. Test every question you plan to ask on stage, twice. Fix the descriptions, not the question.

Docs: [Create a Fabric data agent](https://learn.microsoft.com/fabric/data-science/how-to-create-data-agent)